# FTEC-5320 Lab: Zero Knowledge Proof and Anonymous Trading

### Developed by Hanzhi Xiao and Shu Yang


## The big idea

Public blockchains are transparent by default.

If a normal DeFi contract asks "who is allowed to trade?", the answer is often tied to a public address:

```text
msg.sender = 0x...
```

Zero-knowledge proofs let the application ask a different question:

```text
Does this person know a valid private witness for an allowed public statement?
```

The contract can verify the answer without learning the witness.


## A proof system view

Many modern ZK systems can be described with two main algorithms:

$$
\pi \leftarrow Prove(pk, x, w)
$$

$$
Verify(vk, x, \pi) \in \{0,1\}
$$

Here:

- `x` is public input, visible to the verifier and often visible on-chain
- `w` is the private witness, kept by the prover
- `\pi` is the proof
- `pk` and `vk` are proving and verification keys in many SNARK-style systems

On-chain applications normally do not see `w`.
They only receive public inputs and a proof, then run verification.


## ZK in one formula

A zero-knowledge proof is usually described using:

- a **public statement** `x`
- a **private witness** `w`
- a relation `R(x, w)`

The prover wants to convince the verifier that:

$$
\exists w \; \text{such that} \; R(x, w) = 1
$$

without revealing `w`.

In this lab, the rough meaning is:

$$
\exists \; \text{identity secret and Merkle path}
$$

such that the prover is a member of the classroom group and is allowed to use this trade action once.


## Where ZK appears in Web3

ZK is not only about hiding financial activity.
Common application areas include:

- **scaling**: a rollup proves that many off-chain transactions were executed correctly
- **private identity**: a user proves membership, age, region, or eligibility without revealing full identity
- **private payments and DeFi**: a protocol reduces the link between sender, receiver, and account history
- **governance and voting**: a voter proves the right to vote without revealing the voter's identity
- **credentials and games**: a player proves ownership, progress, or reputation without exposing all account data

This lab focuses on identity and access control inside a trading-like application.


## The three properties

A useful ZK system should satisfy three properties.

**Completeness**

If the statement is true and the prover has a valid witness, the verifier accepts:

$$
\Pr[V(x, \pi) = 1 \mid R(x, w)=1] \approx 1
$$

**Soundness**

If the statement is false, a cheating prover should only succeed with tiny probability:

$$
\Pr[V(x, \pi) = 1 \mid \nexists w: R(x,w)=1] \leq \epsilon
$$

**Zero-knowledge**

The verifier learns that the statement is true, but does not learn the witness.
Informally, the proof transcript can be simulated from public information:

$$
\text{View}_{V}(P(w), x) \approx S(x)
$$


## Membership without revealing identity

In a classroom demo, imagine each eligible participant has a private identity secret:

$$
s = \text{identity secret}
$$

The public group contains commitments, not raw secrets:

$$
C = H(s)
$$

These commitments are placed into a Merkle tree with public root:

$$
root
$$

The proof says, in simplified form:

$$
\exists s, path:
H(s) \in \text{MerkleTree}(root)
$$

The contract learns that the prover is in the group.
It does not learn which member's secret was used.


## Why nullifiers matter

Anonymous systems need a way to prevent reuse.
Otherwise, one valid anonymous member could use the same right many times.

A nullifier is a public one-time tag derived from the private identity and the action scope:

$$
nullifier = H(s, scope)
$$

The important intuition:

- the same identity and same scope produce the same nullifier
- a different scope produces a different nullifier
- observers see the nullifier, but should not recover `s`

So the contract can say:

```text
I still do not know who you are,
but I know this anonymous right has already been used.
```


## How this maps to anonymous trading

For the anonymous path, the public statement is roughly:

$$
x = (root, scope, message, nullifier)
$$

The private witness is roughly:

$$
w = (identity\ secret, Merkle\ path)
$$

The proof convinces the contract that:

$$
R(x,w)=1
$$

meaning:

- the prover is in the classroom group
- the proof is for this trade scope
- the proof is bound to the selected recipient
- the nullifier has not been used before


## A Practice: Anonymous Trading

## Before running

Open this notebook from the repository root. All supporting files are in:

```text
lab_assets/
```

The notebook reads ABI and proof files with paths relative to this notebook folder:

- `lab_assets/abis.json`
- `lab_assets/proof_packet.generated.json`

Node.js is required for local Semaphore identity/proof generation. Install the Node packages once:

```bash
cd lab_assets
npm install
```

For proof generation, the notebook includes a cross-platform Python helper below. It passes environment variables directly to Node, so the same notebook cell works on macOS and Windows.

## Scenario
We simulate an institutional trading desk scenario.

1. **KYC phase**

   Institution A, Institution B, and Institution C prove to the trading desk that they are qualified investors. The trading desk knows who these institutions are at this stage.

2. **Eligibility registration phase**

   The trading desk adds each institution's `identityCommitment` to an on-chain Semaphore group.

3. **Trading phase**

   One institution submits a ZK proof. The contract learns that the proof comes from some qualified group member, but it does not need to learn whether the institution is A, B, or C.

This demo is exactly a simplified version of that flow. It focuses on anonymous authorization: the contract authorizes an action by verifying a proof instead of directly identifying the trader wallet.

### What ZK does not hide in this demo

This demo does not hide everything a production private exchange would care about:

- fixed trade size is public
- desk inventory is public
- recipient address is public
- transaction timing is public

The point is narrower and easier to observe:

**the contract verifies eligibility without directly learning which registered institution used the right.**


In [ ]:
from web3 import Web3
import json
import os
from pathlib import Path

# Fill these before running the lab. Paths below are relative to this notebook folder.
lab_assets_dir = Path("lab_assets")
rpc_url = "https://sepolia.infura.io/v3/infra_key"      # Sepolia RPC URL https://sepolia.infura.io/v3/infra_key

wallet_public_address = ""    # wallet used to send transactions
wallet_private_key = ""       # private key for this wallet; never commit a real key

quote_token_address = "0xc83B0efA5B3F13851DfA11de72EF6AFeF026730c"  # USTUSD
trade_token_address = "0x4E22e9951770b98d4f3B2BA6647f0f40A15A4057"  # USTETH
trade_desk_address = "0xc4D89B20B53Ce26b7Ab9A2F3BFBd1c16d5456B08"   # deployed ClassroomAnonymousTradeDesk address

course_faucet_address = "0x4Cd26B8a999582727789E8236789125D8a9022c5"  # USTFaucet
request_course_tokens = False  # set True if this wallet still needs course tokens

recipient_address = ""        # recipient for anonymous output; must match the proof packet
proof_packet_path = lab_assets_dir / "proof_packet.generated.json"

proof_packet = {
    "merkleTreeDepth": 0,
    "merkleTreeRoot": 0,
    "nullifier": 0,
    "message": 0,
    "scope": 0,
    "points": [0, 0, 0, 0, 0, 0, 0, 0]
}


## Generate the Semaphore identity and proof packet

The terminal commands often differ between macOS/Linux and Windows:

- macOS/Linux usually uses `RPC_URL=... npm run proof`
- Windows PowerShell usually uses `$env:RPC_URL=...; npm run proof`

To avoid that difference, the cells below use Python's `subprocess` and pass environment variables directly to Node. The same notebook cells work on macOS and Windows.

Use this flow:

1. Run `npm install` once in `lab_assets`.
2. Generate a Semaphore identity.
3. Copy the printed `identityCommitment`.
4. The TradeDesk owner calls `addMember(identityCommitment)` on-chain.
5. After the transaction is mined, generate `lab_assets/proof_packet.generated.json`.

If you want to verify your own proof against the shared classroom TradeDesk, use one of these two paths:

1. Send only your `identityCommitment` to Shu. Shu will call `addMember(identityCommitment)` from the owner wallet. Do not send your wallet private key, `semaphore_identity.json`, or proof packet.
2. If you want to run the full owner workflow yourself, deploy `lab_assets/contracts/ClassroomAnonymousTradeDesk.sol`, using the course `USTUSD`, `USTETH`, and the Sepolia Semaphore contract. Then your deployer wallet can call `addMember(identityCommitment)` on your own TradeDesk contract.

In both paths, `identityCommitment` is not your wallet address. It is the public group entry derived from your local Semaphore identity. The secret identity file stays local and is used only to generate the proof packet.


In [ ]:
import subprocess


def run_lab_assets_command(command, extra_env=None):
    """Run a fixed command in lab_assets on macOS or Windows."""
    env = os.environ.copy()
    if extra_env:
        env.update({key: str(value) for key, value in extra_env.items()})

    result = subprocess.run(
        command,
        cwd=str(lab_assets_dir),
        env=env,
        shell=True,
        text=True,
        capture_output=True,
    )

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {command}")

    return result


In [ ]:
# Run once to create lab_assets/semaphore_identity.json.
# Then copy the printed identityCommitment and ask the TradeDesk owner to call addMember(identityCommitment).
generate_identity_in_notebook = False

if generate_identity_in_notebook:
    run_lab_assets_command("npm run identity")
else:
    print("Skipping identity generation. Set generate_identity_in_notebook = True when you need a new identityCommitment.")


In [ ]:
# Run after addMember(identityCommitment) is mined.
# This writes lab_assets/proof_packet.generated.json, which the later proof-check cell reads.
generate_proof_packet_in_notebook = False

if generate_proof_packet_in_notebook:
    if not rpc_url:
        raise ValueError("Fill rpc_url first.")
    if not trade_desk_address:
        raise ValueError("Fill trade_desk_address first.")
    if not recipient_address:
        raise ValueError("Fill recipient_address first.")

    run_lab_assets_command(
        "npm run proof",
        {
            "RPC_URL": rpc_url,
            "TRADE_DESK_ADDRESS": trade_desk_address,
            "RECIPIENT_ADDRESS": recipient_address,
            "PROOF_PACKET_PATH": "proof_packet.generated.json",
        },
    )
else:
    print("Skipping proof generation. Set generate_proof_packet_in_notebook = True after addMember(...) is mined.")


## Connect and inspect contracts

This section checks the deployed contracts and prints the fixed trade terms.


In [ ]:
wallet_public_address = Web3.to_checksum_address(wallet_public_address)
quote_token_address = Web3.to_checksum_address(quote_token_address)
trade_token_address = Web3.to_checksum_address(trade_token_address)
trade_desk_address = Web3.to_checksum_address(trade_desk_address)
course_faucet_address = Web3.to_checksum_address(course_faucet_address)
recipient_address = Web3.to_checksum_address(recipient_address)

web3 = Web3(Web3.HTTPProvider(rpc_url))
print("Connected:", web3.is_connected())
print("Chain ID:", web3.eth.chain_id)
print("Latest block:", web3.eth.get_block_number())

abi_path = lab_assets_dir / "abis.json"
with abi_path.open("r", encoding="utf-8-sig") as f:
    abi_data = json.load(f)

# The ABI bundle includes MockToken, which is enough for standard ERC20 calls
# such as symbol(), decimals(), balanceOf(), approve(), and transferFrom().
erc20_abi = abi_data["MockToken"]
quote_token = web3.eth.contract(address=quote_token_address, abi=erc20_abi)
trade_token = web3.eth.contract(address=trade_token_address, abi=erc20_abi)
trade_desk = web3.eth.contract(address=trade_desk_address, abi=abi_data["ClassroomAnonymousTradeDesk"])

ust_faucet_abi = [
    {
        "inputs": [],
        "name": "requestTokens",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function",
    }
]
course_faucet = web3.eth.contract(address=course_faucet_address, abi=ust_faucet_abi)

quote_symbol = quote_token.functions.symbol().call()
trade_symbol = trade_token.functions.symbol().call()
quote_decimals = quote_token.functions.decimals().call()
trade_decimals = trade_token.functions.decimals().call()

print("Transaction wallet:", wallet_public_address)
print("Quote token:", quote_symbol)
print("Trade token:", trade_symbol)
print("Course faucet:", course_faucet.address)
print("Trade desk:", trade_desk.address)


In [ ]:
fixed_input, fixed_output = trade_desk.functions.previewTrade().call()
group_id = trade_desk.functions.groupId().call()
trade_scope = trade_desk.functions.tradeScope().call()
desk_inventory = trade_desk.functions.inventory().call()

print("Group ID:", group_id)
print("Trade scope:", trade_scope)
print("Fixed input:", fixed_input / 10**quote_decimals, quote_symbol)
print("Fixed output:", fixed_output / 10**trade_decimals, trade_symbol)
print("Desk inventory:", desk_inventory / 10**trade_decimals, trade_symbol)


## Act 1: Public trade exposes the trader

In this act, the transaction wallet performs a normal public trade.

- the wallet pays `USTUSD`
- the wallet receives `USTETH`
- the event log records the trader address directly


In [ ]:
def explorer_tx_url(tx_hash):
    return f"https://sepolia.etherscan.io/tx/{tx_hash.hex()}"


def send_tx(contract_function, value=0, gas_limit=700000):
    tx = contract_function.build_transaction(
        {
            "from": wallet_public_address,
            "gas": gas_limit,
            "gasPrice": web3.eth.gas_price,
            "chainId": web3.eth.chain_id,
            "value": value,
            "nonce": web3.eth.get_transaction_count(wallet_public_address),
        }
    )

    signed_txn = web3.eth.account.sign_transaction(tx, wallet_private_key)
    raw_transaction = signed_txn.rawTransaction if hasattr(signed_txn, "rawTransaction") else signed_txn.raw_transaction
    tx_hash = web3.eth.send_raw_transaction(raw_transaction)
    receipt = web3.eth.wait_for_transaction_receipt(tx_hash)

    print("tx hash:", tx_hash.hex())
    print("explorer:", explorer_tx_url(tx_hash))
    print("status:", receipt.status)
    if receipt.status != 1:
        raise RuntimeError(f"Transaction failed: {tx_hash.hex()}")
    return tx_hash, receipt


In [ ]:
quote_balance_before = quote_token.functions.balanceOf(wallet_public_address).call()
trade_balance_before = trade_token.functions.balanceOf(wallet_public_address).call()
print("Quote balance before:", quote_balance_before / 10**quote_decimals, quote_symbol)
print("Trade balance before:", trade_balance_before / 10**trade_decimals, trade_symbol)

if request_course_tokens:
    print("Requesting USTUSD/USTETH from the course faucet...")
    send_tx(course_faucet.functions.requestTokens())
else:
    print("Skipping faucet request. Set request_course_tokens = True if this wallet needs course tokens.")

quote_balance = quote_token.functions.balanceOf(wallet_public_address).call()
trade_balance = trade_token.functions.balanceOf(wallet_public_address).call()
print("Quote balance now:", quote_balance / 10**quote_decimals, quote_symbol)
print("Trade balance now:", trade_balance / 10**trade_decimals, trade_symbol)

if quote_balance < fixed_input:
    raise ValueError(
        f"Not enough {quote_symbol}. Request course tokens from the USTFaucet or transfer funds to this wallet."
    )

send_tx(quote_token.functions.approve(trade_desk_address, fixed_input))
public_trade_tx_hash, public_trade_receipt = send_tx(trade_desk.functions.publicTrade())


In [ ]:
public_trade_events = trade_desk.events.PublicTrade().process_receipt(public_trade_receipt)
public_trade_events



The public path answers the authorization question with a visible address:

```text
trader = transaction wallet
```


## Act 2: Inspect the proof packet

The anonymous path uses a prepared proof packet.

- `merkleTreeRoot`: the public classroom group state
- `scope`: the action this proof is allowed to authorize
- `message`: the recipient address encoded as an integer
- `nullifier`: the one-time anti-reuse tag
- `points`: the proof data checked by the verifier


In [ ]:
def to_int(value):
    if isinstance(value, str):
        return int(value, 0)
    return int(value)


def normalize_proof_packet(packet):
    required_keys = [
        "merkleTreeDepth",
        "merkleTreeRoot",
        "nullifier",
        "message",
        "scope",
        "points",
    ]
    missing = [key for key in required_keys if key not in packet]
    if missing:
        raise ValueError(f"Proof packet is missing keys: {missing}")
    if len(packet["points"]) != 8:
        raise ValueError("proof_packet['points'] must contain exactly 8 values")

    return {
        "merkleTreeDepth": to_int(packet["merkleTreeDepth"]),
        "merkleTreeRoot": to_int(packet["merkleTreeRoot"]),
        "nullifier": to_int(packet["nullifier"]),
        "message": to_int(packet["message"]),
        "scope": to_int(packet["scope"]),
        "points": [to_int(value) for value in packet["points"]],
    }


path = Path(proof_packet_path)
if path.exists():
    with path.open("r", encoding="utf-8-sig") as f:
        loaded_packet = json.load(f)
    proof_packet = {key: loaded_packet[key] for key in loaded_packet if not key.startswith("_")}

proof_packet = normalize_proof_packet(proof_packet)

expected_message = int(recipient_address, 16)
nullifier_is_used = trade_desk.functions.nullifierUsed(proof_packet["nullifier"]).call()
points_are_placeholder = all(value == 0 for value in proof_packet["points"])

print("Recipient address:", recipient_address)
print("Expected message:", expected_message)
print("Proof message:", proof_packet["message"])
print("Message matches recipient?", expected_message == proof_packet["message"])
print("Desk scope:", trade_scope)
print("Proof scope:", proof_packet["scope"])
print("Scope matches desk?", trade_scope == proof_packet["scope"])
print("Nullifier:", proof_packet["nullifier"])
print("Nullifier already used?", nullifier_is_used)
print("Proof points still placeholder?", points_are_placeholder)

if points_are_placeholder:
    raise ValueError("Generate lab_assets/proof_packet.generated.json with npm run proof before the anonymous trade section.")
if proof_packet["message"] != expected_message:
    raise ValueError("Proof message does not match the selected recipient address.")
if proof_packet["scope"] != trade_scope:
    raise ValueError("Proof scope does not match the deployed trade desk scope.")
if nullifier_is_used:
    raise ValueError("This nullifier has already been used. Use a fresh proof packet.")


## Act 3: Anonymous trade succeeds

Now the contract verifies the proof and sends the fixed output to the recipient.

The wallet still sends the transaction, because someone must pay gas.
But the eligibility check is done by the proof, not by treating the sender address as the identity.


In [ ]:
semaphore_proof = (
    proof_packet["merkleTreeDepth"],
    proof_packet["merkleTreeRoot"],
    proof_packet["nullifier"],
    proof_packet["message"],
    proof_packet["scope"],
    proof_packet["points"]
)

anonymous_tx_hash, anonymous_receipt = send_tx(
    trade_desk.functions.anonymousTrade(recipient_address, semaphore_proof)
)


In [ ]:
anonymous_trade_events = trade_desk.events.AnonymousTrade().process_receipt(anonymous_receipt)
anonymous_trade_events


In [ ]:
recipient_trade_balance = trade_token.functions.balanceOf(recipient_address).call()
print("Recipient trade token balance:", recipient_trade_balance / 10**trade_decimals, trade_symbol)
print("Nullifier used now?", trade_desk.functions.nullifierUsed(proof_packet["nullifier"]).call())


Compare the two event logs in the institutional scenario.

In the KYC phase, the trading desk may know that Institution A, B, and C are qualified investors. The question is what the public contract event reveals when one qualified institution trades.

Public trade event:

```text
trader = visible wallet
```

This is like saying: the on-chain event points directly to the wallet that submitted the trade. If that wallet is known to belong to Institution A, observers can link the trade to Institution A.

Anonymous trade event:

```text
recipient = output address
nullifier = one-time tag
```

This is different. The event shows the output address and the one-time nullifier, but it does not say whether the eligible institution was A, B, or C. The contract only learns: some registered qualified member used this trading right once.

So the demo is not hiding the entire trade. It is changing the authorization question from:

```text
Which wallet are you?
```

to:

```text
Can you prove you are one of the qualified institutions without revealing which one?
```


## Act 4: Reusing the same proof fails

A common question is:

"If the system does not know who I am, why can't I reuse the proof?"

The answer is the nullifier.

Now we try the same anonymous trade again.
The expected result is a revert because the nullifier is already marked as used.


In [ ]:
try:
    replay_tx_hash, replay_receipt = send_tx(
        trade_desk.functions.anonymousTrade(recipient_address, semaphore_proof)
    )
    print("Unexpected success:", replay_tx_hash.hex())
except Exception as exc:
    print("Replay failed as expected.")
    print(type(exc).__name__)
    print(str(exc)[:500])


## Final takeaway

This demo shows the core ZK application pattern:

$$
\text{public identity check}
\quad \longrightarrow \quad
\text{proof-based eligibility check}
$$

The smart contract verifies:

$$
\exists w: R(x,w)=1
$$

but it does not learn the private witness `w`.

In practical terms:

- public trade: "this wallet traded"
- anonymous trade: "someone eligible used this one-time right"
